# Phase 0 — EDA Gate: Olist E-Commerce

**Purpose:** GO/NO-GO check before building any pipeline. See `CONTEXT.md` §7 and `../PORTFOLIO_EDA_SPRINT.md`.

> **Honesty banner:** This project runs a *simulated* RCT on *historical* Olist data. Olist has no native A/B column. Random assignment is by hashed `customer_unique_id` with a fixed seed; any treatment effect is synthetic and labeled `SIMULATED_EFFECT`.

**Go criteria:** ≥50k valid orders · conversion computable with <5% ambiguous status · metric SQL reproducible (DuckDB == pandas) · join orphans <1%.

Numbers below are produced by executing this notebook — none are hand-typed.

In [1]:
import hashlib
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy import stats

RAW = Path("../data/raw/olist")
SEED = 42  # pinned everywhere per AGENTS.md

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
pays = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
custs = pd.read_csv(RAW / "olist_customers_dataset.csv")
print("loaded")

loaded


## 1. Row counts & schema

In [2]:
for name, df in [("orders", orders), ("items", items), ("payments", pays), ("customers", custs)]:
    print(f"{name:10s} rows={len(df):>7d}  cols={list(df.columns)}")

orders     rows=  99441  cols=['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
items      rows= 112650  cols=['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
payments   rows= 103886  cols=['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
customers  rows=  99441  cols=['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


## 2. Order status distribution

`delivered` is the conversion target. Everything else is either terminal-fail (`canceled`, `unavailable`) or in-flight (`shipped`, `invoiced`, `processing`, `created`, `approved`).

In [3]:
print(orders["order_status"].value_counts(dropna=False))
print()
print((orders["order_status"].value_counts(normalize=True) * 100).round(2))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64


## 3. Timestamp coverage

In [4]:
ts = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
print(f"purchase ts: min={ts.min()}  max={ts.max()}  nulls={ts.isna().sum()}")
for col in [c for c in orders.columns if "timestamp" in c or "date" in c]:
    n = orders[col].isna().sum()
    print(f"  {col:35s} nulls={n} ({n/len(orders)*100:.2f}%)")

purchase ts: min=2016-09-04 21:15:19  max=2018-10-17 17:30:18  nulls=0
  order_purchase_timestamp            nulls=0 (0.00%)
  order_delivered_carrier_date        nulls=1783 (1.79%)
  order_delivered_customer_date       nulls=2965 (2.98%)
  order_estimated_delivery_date       nulls=0 (0.00%)


## 4. Join integrity (orphan rows)

Gate: no FK orphan group >1% without explanation.

In [5]:
oid = set(orders["order_id"])
cid = set(custs["customer_id"])
n = len(orders)
print(f"items orphan (no order):    {(~items['order_id'].isin(oid)).sum()}")
print(f"payments orphan (no order): {(~pays['order_id'].isin(oid)).sum()}")
print(f"orders w/o customer:        {(~orders['customer_id'].isin(cid)).sum()}")
wo_pay = (~orders['order_id'].isin(set(pays['order_id']))).sum()
wo_item = (~orders['order_id'].isin(set(items['order_id']))).sum()
print(f"orders w/o any payment:     {wo_pay} ({wo_pay/n*100:.2f}%)")
print(f"orders w/o any item:        {wo_item} ({wo_item/n*100:.2f}%)")

items orphan (no order):    0
payments orphan (no order): 0
orders w/o customer:        0


orders w/o any payment:     1 (0.00%)


orders w/o any item:        775 (0.78%)


## 5. Conversion — overall & by month

Boundary months are sparse/right-censored: the earliest months have a handful of orders, and the final months show ~0% delivered because recent orders are still in-flight, not failures. **The experiment cohort must exclude these boundary months.**

In [6]:
conv = (orders["order_status"] == "delivered").mean()
print(f"overall delivered rate: {conv*100:.2f}%")
om = orders.merge(custs[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
om["ts"] = pd.to_datetime(om["order_purchase_timestamp"], errors="coerce")
om["month"] = om["ts"].dt.to_period("M")
mc = om.assign(d=om["order_status"] == "delivered").groupby("month")["d"].agg(["mean", "count"])
mc["mean"] = (mc["mean"] * 100).round(2)
print(f"months: {mc.index.min()} .. {mc.index.max()}  ({len(mc)} months)")
print(mc.head(4)); print("..."); print(mc.tail(4))

overall delivered rate: 97.02%


months: 2016-09 .. 2018-10  (25 months)
           mean  count
month                 
2016-09   25.00      4
2016-10   81.79    324
2016-12  100.00      1
2017-01   93.75    800
...
          mean  count
month                
2018-07  97.89   6292
2018-08  97.53   6512
2018-09   0.00     16
2018-10   0.00      4


## 6. AOV (average order value)

Sum of `payment_value` per order. Right-skewed continuous metric — strongest A/B candidate.

In [7]:
aov = pays.groupby("order_id")["payment_value"].sum()
print(aov.describe().round(2))

count    99440.00
mean       160.99
std        221.95
min          0.00
25%         62.01
50%        105.29
75%        176.97
max      13664.08
Name: payment_value, dtype: float64


## 7. D7 repeat-purchase feasibility

Person identity = `customer_unique_id` (`customer_id` is per-order). D7 = a 2nd order within 7 days of the 1st.

In [8]:
persons = om["customer_unique_id"].nunique()
ever2 = (om.groupby("customer_unique_id")["order_id"].nunique() >= 2).sum()
first = om.sort_values("ts").groupby("customer_unique_id")["ts"].transform("first")
om["days_since_first"] = (om["ts"] - first).dt.days
d7 = om[(om["days_since_first"] > 0) & (om["days_since_first"] <= 7)]["customer_unique_id"].nunique()
print(f"unique persons:        {persons}")
print(f">=2 orders ever:       {ever2} ({ever2/persons*100:.2f}%)")
print(f"2nd order within 7d:   {d7} ({d7/persons*100:.3f}%)  <- too sparse for a primary metric")

unique persons:        96096
>=2 orders ever:       2997 (3.12%)
2nd order within 7d:   206 (0.214%)  <- too sparse for a primary metric


## 8. Simulated assignment balance (seed=42)

Hash `customer_unique_id` → 50/50. With no treatment effect, delivered rate should be ~equal across variants (a near-null sanity check).

In [9]:
def variant(uid: str) -> int:
    return int(hashlib.md5(f"{uid}-{SEED}".encode()).hexdigest(), 16) % 2

om["variant"] = om["customer_unique_id"].map(variant)
print(om["variant"].value_counts())
print("delivered rate by variant (%):")
print((om.assign(d=om["order_status"] == "delivered").groupby("variant")["d"].mean() * 100).round(3))

variant
0    49866
1    49575
Name: count, dtype: int64
delivered rate by variant (%):
variant
0    96.924
1    97.117
Name: d, dtype: float64


## 9. DuckDB == pandas parity

Every metric will live in `sql/`. Confirm the SQL engine reproduces the pandas number exactly.

In [10]:
con = duckdb.connect()
con.register("orders", orders)
duck = con.execute(
    "SELECT 100.0*SUM(CASE WHEN order_status='delivered' THEN 1 ELSE 0 END)/COUNT(*) FROM orders"
).fetchone()[0]
print(f"pandas={conv*100:.4f}%  duckdb={duck:.4f}%  match={abs(conv*100-duck) < 1e-6}")

pandas=97.0203%  duckdb=97.0203%  match=True


## 10. Power / MDE

Two arms, ~49.7k each, α=0.05, power=0.80. Documents the minimum detectable effect per metric.

In [11]:
n_arm = int(om["variant"].value_counts().min())
za, zb = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
p0 = conv
mde_prop = (za + zb) * np.sqrt(2 * p0 * (1 - p0) / n_arm)
mu, sd = aov.mean(), aov.std()
mde_aov = (za + zb) * sd * np.sqrt(2 / n_arm)
print(f"n/arm = {n_arm}")
print(f"Conversion: baseline={p0*100:.2f}%  MDE={mde_prop*100:.3f}pp (rel {mde_prop/p0*100:.2f}%)")
print(f"AOV:        mean={mu:.2f}  MDE={mde_aov:.2f} BRL (rel {mde_aov/mu*100:.2f}%)")

n/arm = 49575
Conversion: baseline=97.02%  MDE=0.303pp (rel 0.31%)
AOV:        mean=160.99  MDE=3.95 BRL (rel 2.45%)


## Verdict: **GO (with caveats)**

Structural gates pass: ~99.4k orders, ambiguous status <5%, orphans <1%, DuckDB==pandas.

Caveats carried into design:
1. **Conversion is a 97% ceiling metric** — powerable (MDE ~0.3pp at this n) but semantically a *fulfillment* rate, not checkout conversion. Frame honestly or use a funnel step.
2. **AOV** is the strongest continuous primary metric (MDE ~2.5% relative).
3. **D7-within-7d ≈ 0.2%** — too sparse; demote to exploratory.
4. **Cohort window** must exclude boundary months (sparse early, right-censored late).
5. No native A/B — assignment is balanced/near-null; a labeled `SIMULATED_EFFECT` is required for the methodology demo.

Full verdict: `../reports/eda_gate.md`.